In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

# ---------------------------------------------------------
# PATHS (COLAB)
# ---------------------------------------------------------
DATASET_PATH = "/content/drive/MyDrive/Datasets/rice/train"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Datasets/rice/model_rice_disease.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 4


# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------
def load_dataset(path):
    filepaths = []
    labels = []

    for class_name in os.listdir(path):
        class_path = os.path.join(path, class_name)

        if os.path.isdir(class_path):
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".png", ".jpeg")):
                    filepaths.append(os.path.join(class_path, img))
                    labels.append(class_name)

    return pd.DataFrame({"Filepath": filepaths, "Label": labels})


# ---------------------------------------------------------
# Hybrid Model
# ---------------------------------------------------------
def build_hybrid_model(num_classes):

    effnet = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    effnet.trainable = False

    convnext = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    convnext.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))

    x1 = effnet(inputs)
    x2 = convnext(inputs)

    x = tf.keras.layers.Concatenate()([x1, x2])
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)


# ---------------------------------------------------------
# Train
# ---------------------------------------------------------
def train():

    print("Loading dataset...")
    df = load_dataset(DATASET_PATH)

    train_df, val_df = train_test_split(
        df,
        train_size=0.8,
        stratify=df["Label"],
        random_state=42
    )

    train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        horizontal_flip=True
    )

    val_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
    )

    train_images = train_gen.flow_from_dataframe(
        train_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    val_images = val_gen.flow_from_dataframe(
        val_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    print("Classes:", train_images.class_indices)

    model = build_hybrid_model(len(train_images.class_indices))

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    history = model.fit(
        train_images,
        validation_data=val_images,
        epochs=EPOCHS
    )

    print("Saving model...")
    model.save(MODEL_SAVE_PATH)

    # Save plots to Drive
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.legend(["Train", "Val"])
    plt.title("Accuracy")
    plt.savefig("/content/drive/MyDrive/Datasets/rice/accuracy.png")
    plt.close()

    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.legend(["Train", "Val"])
    plt.title("Loss")
    plt.savefig("/content/drive/MyDrive/Datasets/rice/loss.png")
    plt.close()


train()

Loading dataset...
Found 12017 validated image filenames belonging to 10 classes.
Found 3005 validated image filenames belonging to 10 classes.
Classes: {'bacterial_leaf_blight': 0, 'brown_spot': 1, 'healthy': 2, 'leaf_blast': 3, 'leaf_scald': 4, 'narrow_brown_spot': 5, 'neck_blast': 6, 'rice_hispa': 7, 'sheath_blight': 8, 'tungro': 9}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training on GPU 🚀 ...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/4
376/376 ━━━━━━━━━━━━━━━━━━━━ 4754s 12s/step - accuracy: 0.5220 - loss: 1.4966 - val_accuracy: 0.8629 - val_loss: 0.4290
Epoch 2/4
376/376 ━━━━━━━━━━━━━━━━━━━━ 270s 718ms/step - accuracy: 0.8496 - loss: 0.4592 - val_accuracy: 0.9101 - val_loss: 0.2780
Epoch 3/4
376/376 ━━━━━━━━━━━━━━━━━━━━ 259s 688ms/step - accuracy: 0.9004 - loss: 0.3062 - val_accuracy: 0.9314 - val_loss: 0.2223
Epoch 4/4
376/376 ━━━━━━━━━━━━━━━━━━━━ 253s 673ms/step - accuracy: 0.9283 - loss: 0.2274 - val_accuracy: 0.9491 - val_loss: 0.1745
Saving model...
